In [ ]:
!pip install rasterio
!pip install cartopy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 99.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.mask import mask
import geopandas as gpd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import ListedColormap, BoundaryNorm
import os


In [ ]:
def csv_to_tiff_png(csv_path, shapefile_path, tiff_output, png_output, resolution=0.1):
    # Read shapefile
    try:
        boundary = gpd.read_file(shapefile_path)
        if boundary.crs != 'EPSG:4326':
            boundary = boundary.to_crs(epsg=4326)
    except Exception as e:
        raise ValueError(f"Error reading shapefile: {e}")

    # Check shapefile bounds
    bounds = boundary.total_bounds  # [minx, miny, maxx, maxy]
    lon_min, lat_min, lon_max, lat_max = bounds
    print(f"Shapefile bounds for {csv_path}: {bounds}")
    if not all(np.isfinite([lon_min, lat_min, lon_max, lat_max])):
        raise ValueError(f"Invalid shapefile bounds: {bounds}")

    # Read CSV file
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        raise ValueError(f"Error reading CSV file {csv_path}: {e}")

    # Verify required columns
    if 'longitude_latitude' not in df.columns or 'return_level' not in df.columns:
        raise ValueError(f"CSV {csv_path} must contain 'longitude_latitude' and 'return_level' columns")

    # Split longitude_latitude into lon and lat, with validation
    def split_coords(coord):
        if pd.isna(coord) or not isinstance(coord, str):
            return None, None
        parts = coord.split('_')  # Use underscore as separator
        if len(parts) != 2:
            return None, None
        try:
            return float(parts[0]), float(parts[1])
        except ValueError:
            return None, None

    # Apply splitting and filter invalid rows
    coords = df['longitude_latitude'].apply(split_coords)
    df['lon'] = [c[0] for c in coords]
    df['lat'] = [c[1] for c in coords]

    # Print sample of invalid rows
    invalid_rows = df['lon'].isna() | df['lat'].isna()
    if invalid_rows.any():
        print(f"Warning: {invalid_rows.sum()} rows in {csv_path} with invalid longitude_latitude format were skipped")
        print(f"Sample of invalid longitude_latitude values in {csv_path}:")
        print(df[invalid_rows]['longitude_latitude'].head(10))

    # Drop rows with invalid coordinates
    df = df[~invalid_rows]

    if df.empty:
        raise ValueError(f"No valid coordinate data after filtering in {csv_path}")

    # Check coordinate ranges
    print(f"Longitude range in {csv_path}: min={df['lon'].min()}, max={df['lon'].max()}")
    print(f"Latitude range in {csv_path}: min={df['lat'].min()}, max={df['lat'].max()}")

    # Check if coordinates are within shapefile bounds
    if not (df['lon'].between(lon_min, lon_max).any() and df['lat'].between(lat_min, lat_max).any()):
        print(f"Warning: No coordinates in {csv_path} fall within shapefile bounds {bounds}")

    # Check return_level data
    if not np.all(np.isfinite(df['return_level'])):
        print(f"Warning: Non-finite return_level values found in {csv_path}")
        df = df[np.isfinite(df['return_level'])]

    if df.empty:
        raise ValueError(f"No valid return_level data after filtering non-finite values in {csv_path}")

    print(f"Valid data points in {csv_path}: {len(df)}")
    print(f"Return level range in {csv_path}: min={df['return_level'].min()}, max={df['return_level'].max()}")

    # Calculate grid dimensions
    cols = int((lon_max - lon_min) / resolution) + 1
    rows = int((lat_max - lat_min) / resolution) + 1

    # Create empty grid
    grid = np.full((rows, cols), np.nan)

    # Populate grid with return_level values
    for _, row in df.iterrows():
        col = int((row['lon'] - lon_min) / resolution)
        row_idx = int((lat_max - row['lat']) / resolution)
        if 0 <= row_idx < rows and 0 <= col < cols:
            grid[row_idx, col] = row['return_level']

    # Define transform for GeoTIFF
    transform = from_origin(lon_min, lat_max, resolution, resolution)

    # Write to temporary TIFF
    temp_tiff = 'temp_output.tiff'
    try:
        with rasterio.open(
            temp_tiff,
            'w',
            driver='GTiff',
            height=rows,
            width=cols,
            count=1,
            dtype=grid.dtype,
            crs='EPSG:4326',
            transform=transform,
            nodata=np.nan,
        ) as dst:
            dst.write(grid, 1)
    except Exception as e:
        raise ValueError(f"Error writing temporary TIFF file for {csv_path}: {e}")

    # Mask the TIFF to the Philippines boundary
    try:
        with rasterio.open(temp_tiff) as src:
            masked_data, masked_transform = mask(src, boundary.geometry, crop=True, nodata=np.nan)
        # Compute extent from masked_transform
        width, height = masked_data.shape[2], masked_data.shape[1]
        west, south, east, north = (
            masked_transform.c,
            masked_transform.f + masked_transform.e * height,
            masked_transform.c + masked_transform.a * width,
            masked_transform.f,
        )
        if not all(np.isfinite([west, east, south, north])):
            raise ValueError(f"Invalid masked extent: [{west}, {east}, {south}, {north}]")
    except Exception as e:
        raise ValueError(f"Error masking TIFF with shapefile for {csv_path}: {e}")

    # Write final TIFF
    try:
        with rasterio.open(
            tiff_output,
            'w',
            driver='GTiff',
            height=masked_data.shape[1],
            width=masked_data.shape[2],
            count=1,
            dtype=masked_data.dtype,
            crs='EPSG:4326',
            transform=masked_transform,
            nodata=np.nan,
        ) as dst:
            dst.write(masked_data[0], 1)
    except Exception as e:
        raise ValueError(f"Error writing TIFF file for {csv_path}: {e}")

    # Check if masked_data contains valid values
    if np.all(np.isnan(masked_data[0])):
        print(f"Warning: No valid data in masked raster for {csv_path}. Skipping PNG generation.")
        return

    # Define custom colormap and normalization for rainfall
    colors = [
        "#FFFFFF",  # 0 mm
        "#00CED1",  # 1-10 mm
        "#3a5f3a",  # 10-20 mm
        "#73be73",  # 20-30 mm
        "#90ee90",  # 30-40 mm
        "#FF4500",  # 40-50 mm
        "#F0A040",  # 50-70 mm
        "#D07040",  # 70-100 mm
        "#A05050",  # 100-140 mm
        "#803080",  # 140-200 mm
        "#A040A0",  # 200-300 mm
        "#C080C0",  # 300-400 mm
        "#D0A0D0",  # 400-500 mm
        "#E0C0E0",  # 500-600 mm
        "#F0E0F0"   # >600 mm
    ]
    bounds = [0, 1, 10, 20, 30, 40, 50, 70, 100, 140, 200, 300, 400, 500, 600, 1000]
    cmap = ListedColormap(colors)
    norm = BoundaryNorm(bounds, cmap.N)

    # Create PNG visualization with cartopy
    fig = plt.figure(figsize=(12, 8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([115, 130, 5, 20], crs=ccrs.PlateCarree())  # Philippines region
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.set_facecolor('white')  # Set background to white

    # Plot return_level data with custom colormap
    cmap.set_bad('white')  # Make nodata white
    im = ax.imshow(
        masked_data[0],
        cmap=cmap,
        norm=norm,
        transform=ccrs.PlateCarree(),
        extent=[west, east, south, north],
        origin='upper',
    )
    plt.colorbar(im, ax=ax, label='Rainfall (mm)', orientation='vertical', ticks=bounds[:-1])

    # Overlay Philippines boundary
    boundary.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=1)

    # Add gridlines
    ax.gridlines(draw_labels=True, linestyle="--", alpha=0.5)
    # Extract return period from CSV filename for title
    return_period = os.path.basename(csv_path).replace('return_level_', '').replace('.csv', '')
    ax.set_title(f'Rainfall Return Level Map (Philippines, {return_period})')

    try:
        plt.savefig(png_output, dpi=300, bbox_inches='tight')  # Remove transparent=True
    except Exception as e:
        raise ValueError(f"Error saving PNG file for {csv_path}: {e}")
    plt.close()

if __name__ == "__main__":
    # List of CSV files to process
    csv_files = [
        'return_level_5years.csv',
        'return_level_10years.csv',
        'return_level_25years.csv',
        'return_level_50years.csv',
        'return_level_100years.csv',
        'return_level_250years.csv',
        'return_level_500years.csv',
        'return_level_1000years.csv'
    ]
    shapefile = 'PHL_adm0.shp'

    # Process each CSV file
    for csv_file in csv_files:
        tiff_file = csv_file.replace('.csv', '.tiff')
        png_file = csv_file.replace('.csv', '.png')
        print(f"Processing {csv_file}...")
        csv_to_tiff_png(csv_file, shapefile, tiff_file, png_file)
        print(f"Generated {tiff_file} and {png_file}")

Processing return_level_5years.csv...
Shapefile bounds for return_level_5years.csv: [116.94999   5.04917 126.59804  19.39111]
Longitude range in return_level_5years.csv: min=117.05053541969069, max=126.55057887536827
Latitude range in return_level_5years.csv: min=5.150023557551535, max=19.35008851235382
Valid data points in return_level_5years.csv: 2276
Return level range in return_level_5years.csv: min=42.64463342649385, max=410.7338200881292


/usr/local/lib/python3.11/dist-packages/cartopy/io/__init__.py:241: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_coastline.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.11/dist-packages/cartopy/io/__init__.py:241: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_0_boundary_lines_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)


Generated return_level_5years.tiff and return_level_5years.png
Processing return_level_10years.csv...
Shapefile bounds for return_level_10years.csv: [116.94999   5.04917 126.59804  19.39111]
Longitude range in return_level_10years.csv: min=117.05053541969069, max=126.55057887536827
Latitude range in return_level_10years.csv: min=5.150023557551535, max=19.35008851235382
Valid data points in return_level_10years.csv: 2419
Return level range in return_level_10years.csv: min=57.06426529778493, max=709.7832096157802
Generated return_level_10years.tiff and return_level_10years.png
Processing return_level_25years.csv...
Shapefile bounds for return_level_25years.csv: [116.94999   5.04917 126.59804  19.39111]
Longitude range in return_level_25years.csv: min=117.05053541969069, max=126.55057887536827
Latitude range in return_level_25years.csv: min=5.150023557551535, max=19.35008851235382
Valid data points in return_level_25years.csv: 2419
Return level range in return_level_25years.csv: min=68.32